##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

### Which model achieved the highest accuracy?

The best performance was achieved by ResNet50V2 after fine-tuning which had : 91.6% test accuracy.
EfficientNetV2B0 achieved 90.2%, 
while the Custom CNN achieved only 69% validation accuracy, indicating weaker generalization.
Therefore, ResNet50V2 performed best on CIFAR-10.

### Which Model trained faster ? 
The Custom CNN trained the fastest because:
- It has a much smaller architecture.
- It was trained from scratch.
- It contains significantly fewer parameters compared to the pretrained models.

Between the transfer learning models:
- EfficientNetV2B0 trained faster than ResNet50V2
- ResNet50V2 had 23M parameters
- EfficientNetV2B0 had 5M parameters

### How might the architecture explain the differences ? 
- The Custom CNN was trained from scratch and has a simpler architecture, which caused overfitting and lower validation accuracy. 
- EfficientNetV2B0 uses pretrained weights and efficient convolution blocks, allowing it to extract stronger features with fewer parameters and train faster.
- ResNet50V2 uses residual connections that enable deeper networks to learn more complex representations, which explains its higher accuracy but longer training time.

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

# -----------------------------
# 1) Load CIFAR-10
# -----------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

# Keep labels as integers (SparseCategoricalCrossentropy)
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

# Convert images to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

# -----------------------------
# 2) Data augmentation
# -----------------------------
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

# -----------------------------
# 3) Build EfficientNetV2B0 backbone (pretrained)
# -----------------------------
effnet_base = EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
effnet_base.trainable = False  # freeze first (feature extractor)

# -----------------------------
# 4) Full model (preprocess inside model)
# -----------------------------
effnet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input),          # IMPORTANT
    effnet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)
], name="cifar10_efficientnetv2b0")

effnet_model.summary()

# -----------------------------
# 5) Compile + Train (frozen backbone)
# -----------------------------
effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
]

history = effnet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


Model: "cifar10_efficientnetv2b0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing_1 (Resizing)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 7, 7, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,932,122 (22.63 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 5,919,312 (22.58 MB)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 99s 125ms/step - accuracy: 0.6909 - loss: 0.9527 - val_accuracy: 0.8892 - val_loss: 0.3311 - learning_rate: 0.0010
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 85s 120ms/step - accuracy: 0.8136 - loss: 0.5488 - val_accuracy: 0.8976 - val_loss: 0.3045 - learning_rate: 0.0010
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 85s 120ms/step - accuracy: 0.8302 - loss: 0.5012 - val_accuracy: 0.9012 - val_loss: 0.2891 - learning_rate: 0.0010


In [12]:
# -----------------------------
# 6) Test / Evaluate
# -----------------------------
test_loss, test_acc_e = effnet_model.evaluate(x_test, y_test, verbose=0)
print("EfficientNetV2B0 (frozen) test accuracy:", test_acc_e)
print("EfficientNetV2B0 (frozen) test loss    :", test_loss)


EfficientNetV2B0 (frozen) test accuracy: 0.8985000252723694
EfficientNetV2B0 (frozen) test loss    : 0.294201523065567


In [13]:
# Print the total number of layers inside the EfficientNetV2B0 backbone
print("Total layers in EfficientNetV2B0 backbone:", len(effnet_base.layers))

# Filter only layers that actually have learnable parameters (weights/biases)
trainable_layers = [layer for layer in effnet_base.layers if layer.count_params() > 0]

# Print the number of layers that contain learnable parameters "Depth of the Model"
print("Layers with learnable parameters (depth):", len(trainable_layers))


Total layers in EfficientNetV2B0 backbone: 270
Layers with learnable parameters (depth): 150


In [14]:
# Listing all layers that have learnable parameters (trainable_layers)
# Each layer will be printed with:
# (index in the filtered list, layer name, number of parameters)
for i, layer in enumerate(trainable_layers):
    print(i, layer.name, layer.count_params())


0 stem_conv 864
1 stem_bn 128
2 block1a_project_conv 4608
3 block1a_project_bn 64
4 block2a_expand_conv 9216
5 block2a_expand_bn 256
6 block2a_project_conv 2048
7 block2a_project_bn 128
8 block2b_expand_conv 36864
9 block2b_expand_bn 512
10 block2b_project_conv 4096
11 block2b_project_bn 128
12 block3a_expand_conv 36864
13 block3a_expand_bn 512
14 block3a_project_conv 6144
15 block3a_project_bn 192
16 block3b_expand_conv 82944
17 block3b_expand_bn 768
18 block3b_project_conv 9216
19 block3b_project_bn 192
20 block4a_expand_conv 9216
21 block4a_expand_bn 768
22 block4a_dwconv2 1728
23 block4a_bn 768
24 block4a_se_reduce 2316
25 block4a_se_expand 2496
26 block4a_project_conv 18432
27 block4a_project_bn 384
28 block4b_expand_conv 36864
29 block4b_expand_bn 1536
30 block4b_dwconv2 3456
31 block4b_bn 1536
32 block4b_se_reduce 9240
33 block4b_se_expand 9600
34 block4b_project_conv 36864
35 block4b_project_bn 384
36 block4c_expand_conv 36864
37 block4c_expand_bn 1536
38 block4c_dwconv2 3456
3

In [15]:
# -----------------------------
#Fine-tune last layers
# -----------------------------
effnet_base.trainable = True
for layer in effnet_base.layers[:-30]:
    layer.trainable = False

print("Trainable layers in backbone:", sum(l.trainable for l in effnet_base.layers), "/", len(effnet_base.layers))

effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_ft = effnet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_ft, test_acc_ft = effnet_model.evaluate(x_test, y_test, verbose=0)
print("EfficientNetV2B0 (fine-tuned) test accuracy:", test_acc_ft)
print("EfficientNetV2B0 (fine-tuned) test loss    :", test_loss_ft)

Trainable layers in backbone: 30 / 270
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 117s 146ms/step - accuracy: 0.8051 - loss: 0.5844 - val_accuracy: 0.8918 - val_loss: 0.3245
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 100s 142ms/step - accuracy: 0.8322 - loss: 0.5032 - val_accuracy: 0.9000 - val_loss: 0.3036
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 100s 142ms/step - accuracy: 0.8394 - loss: 0.4770 - val_accuracy: 0.9026 - val_loss: 0.2907
EfficientNetV2B0 (fine-tuned) test accuracy: 0.9018999934196472
EfficientNetV2B0 (fine-tuned) test loss    : 0.293972909450531
